# Target trial emulation

In [1]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import statsmodels.api as sm

from econml.dml import CausalForestDML # Causal forest with double machine learning
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier 

## Define mRS outcome target and cutoff for early treatment

In [2]:
MRS_TARGET = 2 # Model will set y as being less than or equal to this value

CUTOFF = 270

## Define adjustment fields

In [3]:
DESCRIPTIVE_FEATURES = [
    'prior_disability',
    'stroke_severity',
    'age',
    'congestive_heart_failure',
    'hypertension',
    'diabetes',
    'afib_anticoagulant',
    'any_afib_diagnosis',
]

MATCHING_FEATURES = [
    'prior_disability',
    'stroke_severity',
    'age',
    'afib_anticoagulant',
    'any_afib_diagnosis',
]

ALL_FEATURES = list(set(DESCRIPTIVE_FEATURES + MATCHING_FEATURES + 
                        ['onset_to_thrombolysis_time', 'onset_to_thrombectomy_time', 'discharge_disability']))

## Load and filter data

In [4]:
data = pd.read_csv("../../data/sam3/cleaned_data.csv", low_memory=False)

# Calculate onset_to_scan
data["onset_to_scan_time"] = data["onset_to_arrival_time"] - data["arrival_to_scan_time"]

print(f"Initial data shape: {data.shape}")

# Remove rows with missing discharge disability
data = data.dropna(subset=["discharge_disability"])

# Keep only infarction cases
data = data[data["infarction"] == 1]

# Keep only onset to arrival < 360 mins
data = data[data["onset_to_arrival_time"] < 360]

# Keep only perfusion_imaging_used == 0
data = data[data["perfusion_imaging_used"] == 0]

# Limit to stroke severity  > 5 
data = data[((data["stroke_severity"] > 5) | data["lvo"] == 1)]

# Keep only teams with at least 10 onset_to_thrombectomy_time
team_counts = data.groupby("stroke_team")["onset_to_thrombectomy_time"].count()
valid_teams = team_counts[team_counts >= 10].index
data = data[data["stroke_team"].isin(valid_teams)]

# Missing treatment times -> sentinel
data["onset_to_thrombolysis_time"] = data["onset_to_thrombolysis_time"].fillna(99999)
data["onset_to_thrombectomy_time"] = data["onset_to_thrombectomy_time"].fillna(99999)

# Remove rows where thrombolysis is after thrombectomy (if both are present)
both_treatments = (
    (data["onset_to_thrombolysis_time"] < 99999)
    & (data["onset_to_thrombectomy_time"] < 99999)
)
thrombolysis_after_thrombectomy = (
    data["onset_to_thrombolysis_time"] > data["onset_to_thrombectomy_time"]
)
data = data[~(both_treatments & thrombolysis_after_thrombectomy)]

# Exclude late thrombectomy (> 720) unless sentinel
data = data[
    (data["onset_to_thrombectomy_time"] < 720)
    | (data["onset_to_thrombectomy_time"] == 99999)
]

# Exclude late thrombolysis (> 360) unless sentinel
data = data[
    (data["onset_to_thrombolysis_time"] < 360)
    | (data["onset_to_thrombolysis_time"] == 99999)
]

print(f"Final data shape: {data.shape}")

# Select features 
data = data[ALL_FEATURES]

# Treatment counts
num_thrombectomy = (data["onset_to_thrombectomy_time"] != 99999).sum()
print(f"Number of patients with thrombectomy: {num_thrombectomy}")

num_both = (
    (data["onset_to_thrombolysis_time"] != 99999)
    & (data["onset_to_thrombectomy_time"] != 99999)
).sum()
print(f"Number of patients with both thrombolysis and thrombectomy: {num_both}")

num_only_thrombectomy = (
    (data["onset_to_thrombectomy_time"] != 99999)
    & (data["onset_to_thrombolysis_time"] == 99999)
).sum()
print(f"Number of patients with only thrombectomy: {num_only_thrombectomy}")

num_thrombolysis = (data["onset_to_thrombolysis_time"] != 99999).sum()
print(f"Number of patients with thrombolysis: {num_thrombolysis}")

num_only_thrombolysis = (
    (data["onset_to_thrombolysis_time"] != 99999)
    & (data["onset_to_thrombectomy_time"] == 99999)
).sum()
print(f"Number of patients with only thrombolysis: {num_only_thrombolysis}")

Initial data shape: (452863, 71)
Final data shape: (59570, 71)
Number of patients with thrombectomy: 4609
Number of patients with both thrombolysis and thrombectomy: 2876
Number of patients with only thrombectomy: 1733
Number of patients with thrombolysis: 26923
Number of patients with only thrombolysis: 24047


Add treatment columns and patient ID (for honest splitting) for Causal Forest

In [5]:
# Add treatment columns

TIME_THROMBOLYSIS = "onset_to_thrombolysis_time"
TIME_THROMBECTOMY = "onset_to_thrombectomy_time"

def add_treatment_columns(data):
    """
    Add treatment column based on use and time of thrombolysis only.
    Use CUTOFF to determine whether treatment is early or late.
    Sentinel value of 99999 means no treatment. 
    """

    data["treatment_group"] = "neither"
    # Early thrombolysis only
    data.loc[(data[TIME_THROMBOLYSIS] <= CUTOFF) & (data[TIME_THROMBECTOMY] == 99999),
             "treatment_group"] = "early_thrombolysis_only"
    # Late thrombolysis only
    data.loc[(data[TIME_THROMBOLYSIS] > CUTOFF) & (data[TIME_THROMBOLYSIS] != 99999) &
             (data[TIME_THROMBECTOMY] == 99999),
             "treatment_group"] = "late_thrombolysis_only"
    # Early thrombectomy only
    data.loc[(data[TIME_THROMBOLYSIS] == 99999) & (data[TIME_THROMBECTOMY] < CUTOFF), 
             "treatment_group"] = "early_thrombectomy_only"
    # Late thrombectomy only
    data.loc[(data[TIME_THROMBOLYSIS] == 99999) & (data[TIME_THROMBECTOMY] >= CUTOFF) &
             (data[TIME_THROMBECTOMY] != 99999),
             "treatment_group"] = "late_thrombectomy_only"
    # Early thrombolysis and early thrombectomy
    data.loc[(data[TIME_THROMBOLYSIS] <= CUTOFF) & (data[TIME_THROMBECTOMY] < CUTOFF),
             "treatment_group"] = "early_both"
    # Early thrombolysis and late thrombectomy
    data.loc[(data[TIME_THROMBOLYSIS] <= CUTOFF) & (data[TIME_THROMBECTOMY] >= CUTOFF) & 
             (data[TIME_THROMBECTOMY] != 99999), 
             "treatment_group"] = "early_thrombolysis_late_thrombectomy"
    # Late both thrombolysis and thrombectomy
    data.loc[(data[TIME_THROMBOLYSIS] > CUTOFF) &(data[TIME_THROMBOLYSIS] != 99999) &
             (data[TIME_THROMBECTOMY] > CUTOFF) & (data[TIME_THROMBECTOMY] != 99999),
             "treatment_group"] = "late_both"

    return data

data = add_treatment_columns(data)

data["patient_id"] = np.arange(len(data))

In [6]:
# Create treatment groups depending  on whether no treatment, thrombolysis only, thrombectomy only,
# or both treatments. Ignore time to treatment.

def add_treatment_type(data):
    """
    Add treatment column based on use and time of thrombolysis only.
    Use CUTOFF to determine whether treatment is early or late.
    Sentinel value of 99999 means no treatment. 
    """

    data["treatment_type"] = "neither"
    # Thrombolysis only
    data.loc[(data[TIME_THROMBOLYSIS] != 99999) & (data[TIME_THROMBECTOMY] == 99999),
             "treatment_type"] = "thrombolysis_only"
    # Thrombectomy only
    data.loc[(data[TIME_THROMBOLYSIS] == 99999) & (data[TIME_THROMBECTOMY] != 99999),
             "treatment_type"] = "thrombectomy_only"
    # Both treatments
    data.loc[(data[TIME_THROMBOLYSIS] != 99999) & (data[TIME_THROMBECTOMY] != 99999),
             "treatment_type"] = "both"

    return data

data = add_treatment_type(data)

descriptive_data = data[DESCRIPTIVE_FEATURES + ['treatment_type']].copy(deep=True)

# Produce descriptive stats by treatment type
descriptive_stats = descriptive_data.groupby('treatment_type').describe()
descriptive_stats

prior_disability                                          \
                             count      mean       std  min  25%  50%  75%   
treatment_type                                                               
both                        2876.0  0.372045  0.700785  0.0  0.0  0.0  1.0   
neither                    30914.0  1.842499  1.595412  0.0  0.0  2.0  3.0   
thrombectomy_only           1733.0  0.609348  0.824190  0.0  0.0  0.0  1.0   
thrombolysis_only          24047.0  0.984821  1.312634  0.0  0.0  0.0  2.0   

                       stroke_severity             ... afib_anticoagulant  \
                   max           count       mean  ...                75%   
treatment_type                                     ...                      
both               5.0          2876.0  16.107093  ...                0.0   
neither            5.0         30877.0  12.865401  ...                1.0   
thrombectomy_only  5.0          1732.0  16.099885  ...                1.0   
thrombolysis_only  5.0         24031.0  12.702967  ...                0.0   

                       any_afib_diagnosis                                     \
                   max              count      mean       std  min  25%  50%   
treatment_type                                                                 
both               1.0             2876.0  0.243394  0.429205  0.0  0.0  0.0   
neither            1.0            30914.0  0.393770  0.488593  0.0  0.0  0.0   
thrombectomy_only  1.0             1733.0  0.518177  0.499814  0.0  0.0  1.0   
thrombolysis_only  1.0            24047.0  0.213831  0.410018  0.0  0.0  0.0   

                             
                   75%  max  
treatment_type               
both               0.0  1.0  
neither            1.0  1.0  
thrombectomy_only  1.0  1.0  
thrombolysis_only  0.0  1.0  

[4 rows x 64 columns]

In [7]:
ordered_arms = [
    "neither",
    "thrombolysis_only",
    "thrombectomy_only",
    "both"
]

# Format the descriptive stats so it shows mean ± sd and n in brackets.
# Have features as rows and treatment groups as columns.
feature_order = [
    feature
    for feature in pd.Index(descriptive_stats.columns.get_level_values(0)).unique()
    if feature != "treatment_type"
]

formatted_descriptive_stats = pd.DataFrame(index=feature_order)

for group in ordered_arms:
    if group in descriptive_stats.index:
        group_stats = descriptive_stats.loc[group]
        formatted_descriptive_stats[group] = [
            f"{group_stats[(feature, 'mean')]:.2f} ± {group_stats[(feature, 'std')]:.2f} "
            f"(n={int(group_stats[(feature, 'count')])})"
            for feature in feature_order
    ]

formatted_descriptive_stats.index.name = "feature"
# Replace _ with space in the index and column names
formatted_descriptive_stats.index = formatted_descriptive_stats.index.str.replace("_", " ")
formatted_descriptive_stats.columns = formatted_descriptive_stats.columns.str.replace("_", " ")
formatted_descriptive_stats


,neither,thrombolysis only,thrombectomy only,both
feature,,,,
prior disability,1.84 ± 1.60 (n=30914),0.98 ± 1.31 (n=24047),0.61 ± 0.82 (n=1733),0.37 ± 0.70 (n=2876)
stroke severity,12.87 ± 6.75 (n=30877),12.70 ± 6.17 (n=24031),16.10 ± 6.14 (n=1732),16.11 ± 6.03 (n=2876)
age,78.51 ± 12.36 (n=30914),74.22 ± 13.29 (n=24047),73.34 ± 12.90 (n=1733),70.82 ± 13.13 (n=2876)
congestive heart failure,0.09 ± 0.29 (n=30914),0.05 ± 0.22 (n=24047),0.08 ± 0.28 (n=1733),0.04 ± 0.20 (n=2876)
hypertension,0.59 ± 0.49 (n=30914),0.55 ± 0.50 (n=24047),0.55 ± 0.50 (n=1733),0.50 ± 0.50 (n=2876)
diabetes,0.26 ± 0.44 (n=30914),0.21 ± 0.41 (n=24047),0.19 ± 0.39 (n=1733),0.16 ± 0.37 (n=2876)
afib anticoagulant,0.29 ± 0.45 (n=30914),0.03 ± 0.18 (n=24047),0.46 ± 0.50 (n=1733),0.03 ± 0.17 (n=2876)
any afib diagnosis,0.39 ± 0.49 (n=30914),0.21 ± 0.41 (n=24047),0.52 ± 0.50 (n=1733),0.24 ± 0.43 (n=2876)


In [8]:
# Summary mean, sd, and time time to thrombolysis and thrombectomy by treatment type
summary_times = data.groupby("treatment_group")[[TIME_THROMBOLYSIS, TIME_THROMBECTOMY]].agg(["mean", "std", "count"])
summary_times.round(1)

onset_to_thrombolysis_time               \
                                                           mean   std  count   
treatment_group                                                                
early_both                                                142.5  45.6   2570   
early_thrombectomy_only                                 99999.0   0.0   1207   
early_thrombolysis_late_thrombectomy                      209.8  50.1    226   
early_thrombolysis_only                                   163.8  50.4  23074   
late_both                                                 305.8  25.1     80   
late_thrombectomy_only                                  99999.0   0.0    526   
late_thrombolysis_only                                    298.4  23.6    973   
neither                                                 99999.0   0.0  30914   

                                     onset_to_thrombectomy_time                
                                                           mean    std  count  
treatment_group                                                                
early_both                                                163.4   48.9   2570  
early_thrombectomy_only                                   163.2   53.6   1207  
early_thrombolysis_late_thrombectomy                      365.0  124.0    226  
early_thrombolysis_only                                 99999.0    0.0  23074  
late_both                                                 333.4   55.3     80  
late_thrombectomy_only                                    385.5  100.3    526  
late_thrombolysis_only                                  99999.0    0.0    973  
neither                                                 99999.0    0.0  30914

In [9]:
## Target trial emulation